# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os
if not os.path.exists('FlyRank_AI'):
    !git clone -q https://github.com/AhmedMahmoud-123/FlyRank_AI.git
os.chdir('FlyRank_AI')
!python scripts/01_prepare_features.py

import pandas as pd
df = pd.read_csv('data/processed/refresh_feature_vector.csv')
print(f'{len(df):,} rows loaded')

Prepared 30,000 rows from 30,000 raw rows
Wrote /content/FlyRank_AI/data/processed/refresh_feature_vector.csv
30,000 rows loaded


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Method:** Logistic Regression first, then Random Forest.

My lane predicts `is_declining_label` — a yes/no outcome with an observed label — which per
the toolkit's question-shape table means "start readable, then go stronger." Logistic
Regression gives interpretable coefficients I can sanity-check against domain intuition
before trusting anything fancier. Random Forest is the natural next step because it can
pick up non-linear interactions (e.g. staleness only matters *combined with* prior
visibility) that a linear model can't. I'm skipping Gradient Boosting for now — the ML-07
baseline already gets meaningful lift from simple rules, so added complexity needs to earn
its place per training-honest-models, not be assumed.

In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

feature_cols = ['impressions_prev_30d', 'avg_position', 'days_since_last_update',
                'log_impressions_90d', 'has_clicks', 'measurable_opportunity']
X = df[feature_cols]
y = df['is_declining_label']
print(X.isna().sum())  # confirm no blanks slipped through the prep script

impressions_prev_30d      0
avg_position              0
days_since_last_update    0
log_impressions_90d       0
has_clicks                0
measurable_opportunity    0
dtype: int64


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Split:** GroupShuffleSplit on `client_id`, not a random row split.

Per the data dictionary, `client_id` is a pseudonym for grouping only, and content items
from the same client share client-level patterns (their typical position range, their
update cadence). A random split lets those patterns leak across train/test, inflating the
score in a way that won't hold up on a client the model has never seen — the same leakage
notebook 02 demonstrated. GroupShuffleSplit keeps every row from one client entirely on one
side of the split, matching the honest-validation approach from `hunting-leakage-and-validating`.

In [3]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df['client_id']))
X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

print('clients in train:', df.iloc[train_idx]['client_id'].nunique())
print('clients in test: ', df.iloc[test_idx]['client_id'].nunique())
print('base rate (test):', y_te.mean().round(3))

clients in train: 24
clients in test:  8
base rate (test): 0.517


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

**Comparison:** same test split (by client), same metric (precision@K), same base rate
reference as the Week-4 baseline. The baseline's ranked queue is recomputed here on the
*test* rows only, so it's a fair head-to-head, not a different data slice.

In [4]:
import numpy as np

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# --- baseline (Week 4 rule), recomputed on the test split only ---
test_df = df.iloc[test_idx].copy()
visible = (test_df['impressions_prev_30d'] >= 500).astype(int)
stale = (test_df['days_since_last_update'] >= 180).astype(int)
slipping = (test_df['avg_position'] > test_df.loc[test_df['avg_position'] > 0, 'avg_position'].median()).astype(int)
baseline_score = visible * (stale + slipping) * test_df['impressions_prev_30d']

# --- logistic regression ---
logreg = LogisticRegression(max_iter=1000, random_state=42).fit(X_tr, y_tr)
logreg_score = logreg.predict_proba(X_te)[:, 1]

# --- random forest ---
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)
rf_score = rf.predict_proba(X_te)[:, 1]

results = []
for name, scores in [('baseline_rule', baseline_score), ('logistic_regression', logreg_score), ('random_forest', rf_score)]:
    row = {'model': name}
    for k in [20, 50, 100]:
        row[f'precision@{k}'] = round(precision_at_k(scores, y_te.values, k), 3)
    results.append(row)

comparison = pd.DataFrame(results)
comparison['base_rate'] = round(y_te.mean(), 3)
comparison

,model,precision@20,precision@50,precision@100,base_rate
0,baseline_rule,0.55,0.58,0.51,0.517
1,logistic_regression,0.55,0.60,0.58,0.517
2,random_forest,0.95,0.82,0.83,0.517


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [5]:
# feature importance from the model that won above (swap rf/logreg as needed)
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(importances)

# permutation importance as a cross-check (importances above can overstate high-cardinality features)
from sklearn.inspection import permutation_importance
perm = permutation_importance(rf, X_te, y_te, n_repeats=10, random_state=42, n_jobs=-1)
pd.Series(perm.importances_mean, index=feature_cols).sort_values(ascending=False)

impressions_prev_30d      0.366559
log_impressions_90d       0.292662
avg_position              0.248375
days_since_last_update    0.070414
measurable_opportunity    0.011363
has_clicks                0.010627
dtype: float64


/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


,0
impressions_prev_30d,0.159550
log_impressions_90d,0.066339
avg_position,0.022319
has_clicks,0.014645
measurable_opportunity,0.007351
days_since_last_update,0.006831


In [6]:
# 3 concrete wrong cases
test_df['rf_score'] = rf_score
test_df['rf_pred'] = (rf_score >= 0.5).astype(int)
wrong = test_df[test_df['rf_pred'] != test_df['is_declining_label']]
wrong[['content_id', 'impressions_prev_30d', 'avg_position', 'days_since_last_update',
       'is_declining_label', 'rf_score']].sample(3, random_state=42)

,content_id,impressions_prev_30d,avg_position,days_since_last_update,is_declining_label,rf_score
17688,content_e734ce8f057a,591,12.4,13,0,0.925
8639,content_cceb86aa5a80,5,12.9,20,0,0.695
25333,content_efbf824eed7c,13,9.0,8,0,0.960


**Where the model is wrong:** [fill after running — e.g. "false negatives cluster around
pages with strong `avg_position` but declining impressions, because the model leans heavily
on position and under-weights the impression trend within the safe feature set"].

**What it leans on:** top features were [name them from the importances table] — [sanity
check: does the top feature plausibly relate to decline, or is the score suspiciously
perfect, which would suggest leakage?].

**Three wrong cases:** [describe the three sampled rows above — what made each hard, not just
that the model missed].

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.